# 02 · Modelo y métricas de rendimiento

Este notebook entrena y evalúa el clasificador de cada comparativa a partir del dataset generado por **01 · Preparación de datos**:

| Comparativa | Clase 0 | Clase 1 |
|---|---|---|
| `1Y_vs_Ctrl` | Ctrl (DCL estable) | 1Y (imagen a 12 meses del diagnóstico de EA) |
| `2Y_vs_Ctrl` | Ctrl (DCL estable) | 2Y (imagen a 24 meses del diagnóstico de EA) |
| `1Y_vs_2Y` | 1Y | 2Y |

Pasos:
1. Definir el pipeline de clasificación.
2. Seguir un fold paso a paso.
3. Estimar el rendimiento con validación cruzada repetida y anidada, y graficarlo.
4. Entrenar el modelo final de 1Y_vs_Ctrl y 2Y_vs_Ctrl.
5. Medir la estabilidad de sus variables.
6. Medir el efecto de reducir el número de variables.
7. Exportar los modelos para inferencia.

Las figuras se guardan en `img/` (PNG a 300 dpi). Cuando una celda genera varias figuras del mismo tipo, se guardan sin mostrarse y la celda indica sus rutas; en el notebook se muestra una figura que las reúne.

Las pruebas de rigor (permutación, líneas base, algoritmos alternativos, horizontes cruzados y análisis longitudinal) están en **03 · Evaluación de rigor**.

## 0. Configuración y carga de datos

Se cargan el dataset procesado y la demografía (edad y sexo por imagen), y se definen los parámetros de la validación cruzada.

Las advertencias se desactivan, incluida la variable de entorno que heredan los procesos que entrenan en paralelo. Así la salida de las celdas muestra solo los resultados.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Los procesos de entrenamiento en paralelo (joblib) heredan esta variable y tampoco imprimen advertencias
import os
os.environ['PYTHONWARNINGS'] = 'ignore'

import joblib
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.base import clone
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, cross_val_predict
from ml_core import (COVARIATES, METRICS, THRESHOLDS, GRID_LR, DESCRIPTOR_ES, build_pipeline, prepare_comparison,
                     outer_splits, tune_in_fold, evaluate_nested, summarize, selected_features, classification_metrics,
                     best_threshold, export_pipeline, load_region_names, split_feature, comparativa_label, save_figure,
                     plot_pipeline_diagram, plot_nested_cv_diagram, plot_fold_anatomy, plot_auc_by_fold, plot_metrics_panel,
                     plot_mean_roc, plot_oof_confusion, plot_coefficients, plot_feature_elimination, plot_shared_variables)

DIR_DATOS = './datos_procesados'
DIR_EXPORTACION = './exported_pipelines'
RUTAS_COHORTES = {1: './subjects/ADNI_1Y/', 2: './subjects/ADNI_2Y/', 0: './subjects/MCI_Constant_FreeSurfer/'}
# Nombres de las regiones: los de FreeSurfer y su traducción al español (ml_core/regions.py)
NOMBRES_REGION, NOMBRES_REGION_ES = load_region_names('./FreeSurferColorLUT.csv')

# Validación cruzada: 5 folds externos x 5 repeticiones con semillas distintas
N_REPEATS, N_FOLDS = 5, 5
SEED = 0
# Remuestreos bootstrap para la estabilidad de las variables
N_BOOT = 500

COMPARATIVAS = {'1Y_vs_Ctrl': [0, 1], '2Y_vs_Ctrl': [0, 2], '1Y_vs_2Y': [1, 2]}
CLASES_NOMBRES = {'1Y_vs_Ctrl': ['Ctrl', '1Y'], '2Y_vs_Ctrl': ['Ctrl', '2Y'], '1Y_vs_2Y': ['1Y', '2Y']}
# Clase con la que se estima el efecto de edad y sexo: Ctrl (y = 0) en las comparativas
# contra Ctrl; todo el train en 1Y vs 2Y, donde no hay grupo de referencia
REFERENCIA_AJUSTE = {'1Y_vs_Ctrl': 0, '2Y_vs_Ctrl': 0, '1Y_vs_2Y': None}

df = pd.read_csv(os.path.join(DIR_DATOS, 'dataset_procesado.csv'))
demografia = pd.read_csv(os.path.join(DIR_DATOS, 'demografia.csv'))
assert (df['PatientID'] == demografia['PatientID']).all() and (df['Target'] == demografia['Target']).all()
print(f"Dataset: {df.shape[0]} imágenes × {df.shape[1] - 3} características de forma")

## 1. El pipeline de clasificación

Todos los pasos forman un único `Pipeline` de scikit-learn. En cada fold de validación, **todos se ajustan solo con los datos de entrenamiento** y después se aplican, ya fijos, a los de prueba. Así ninguna estadística del conjunto de prueba (medianas, correlaciones, variables seleccionadas, coeficientes) influye en el modelo que se evalúa con él.

| Paso | Qué hace | Detalle |
|---|---|---|
| `ajuste` | Resta a cada característica el efecto lineal de la edad y el sexo | Los conversores son ~4 años mayores que los controles. El efecto se estima con los controles del train, para no restar la señal de la enfermedad |
| `varianza` | Elimina características constantes | Protección: tras el ajuste no debería quedar ninguna |
| `escalado` | `RobustScaler`: resta la mediana y divide entre el rango intercuartílico | Pone todas las características en escala comparable sin que los valores extremos dominen |
| `correlacion` | Elimina cada característica con \|r\| > 0.85 respecto a una anterior | Quita redundancia (p. ej. `MeshVolume` y `VoxelVolume` de la misma región) |
| `seleccion` | Lasso: regresión logística con penalización L1; conserva las variables con coeficiente no nulo | Reduce ~1200 variables a unas decenas. `C` controla cuántas sobreviven |
| `clasificador` | Regresión logística L2 con `class_weight='balanced'` | Modelo interpretable: cada coeficiente es el peso de una variable. `balanced` compensa el desbalance de clases |

El grid de hiperparámetros explora la `C` de Lasso y la `C` de la regresión final.

In [ ]:
plot_pipeline_diagram()

pipeline = build_pipeline(reference_class=0)
print("Pasos del pipeline:", [nombre for nombre, _ in pipeline.steps])
print("Grid de hiperparámetros:", GRID_LR)

## 2. Anatomía de un fold

Antes de evaluar los 25 folds de cada comparativa, se sigue uno paso a paso (el primero de 1Y_vs_Ctrl):

1. **Partición externa** con `StratifiedGroupKFold`: estratificada (misma proporción de clases en train y test) y agrupada por `Sujeto_Real` (ninguna persona queda en ambos lados).
2. **Validación interna** (`tune_in_fold`), solo con el train externo:
   - Un `GridSearchCV` de 5 folds, también agrupado por sujeto, elige los hiperparámetros por AUC.
   - Con las predicciones fuera de fold del propio train se elige el **umbral de decisión** que maximiza el F1-macro.
3. El modelo con esos hiperparámetros se reentrena con todo el train externo y **se evalúa una sola vez** en el test externo, con ese umbral.

In [ ]:
plot_nested_cv_diagram(n_folds=N_FOLDS)

Abajo se ejecuta ese proceso para un fold. La figura muestra:
- **Izquierda (barras):** cuántas características quedan después de cada paso del pipeline, ajustado con el train de este fold.
- **Derecha (línea):** el F1-macro de la validación interna para cada umbral posible. El punto y la línea punteada marcan el umbral elegido.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

X, y, groups = prepare_comparison(df, demografia, COMPARATIVAS['1Y_vs_Ctrl'])
r, f, idx_train, idx_test = outer_splits(y, groups, N_REPEATS, N_FOLDS, SEED)[0]

print(f"Train: {len(idx_train)} imágenes ({y[idx_train].mean():.0%} clase 1) | "
      f"Test: {len(idx_test)} imágenes ({y[idx_test].mean():.0%} clase 1)")
assert set(groups[idx_train]).isdisjoint(groups[idx_test]), "Un sujeto quedó en train y test"
print("Ningún sujeto aparece en train y test a la vez.\n")

model, params, threshold = tune_in_fold(build_pipeline(reference_class=0), GRID_LR,
                                        X.iloc[idx_train], y[idx_train], groups[idx_train], SEED)
print(f"Hiperparámetros elegidos en la validación interna: {params}")
print(f"Umbral elegido en la validación interna: {threshold:.2f}")

# Número de columnas que deja cada paso (ajustado con el train) al transformar el train
X_paso = X.iloc[idx_train]
columnas_por_paso = [('entrada', X_paso.shape[1])]
for nombre, paso in model.steps[:-1]:
    X_paso = paso.transform(X_paso)
    columnas_por_paso.append((nombre, X_paso.shape[1]))

# F1-macro de la validación interna para cada umbral (las mismas predicciones fuera de fold que usa tune_in_fold)
inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
probs_inner = cross_val_predict(clone(model), X.iloc[idx_train], y[idx_train], groups=groups[idx_train],
                                cv=inner_cv, method='predict_proba')[:, 1]
f1_por_umbral = [f1_score(y[idx_train], (probs_inner >= t).astype(int), average='macro') for t in THRESHOLDS]
plot_fold_anatomy(columnas_por_paso, THRESHOLDS, f1_por_umbral, threshold)

probs_test = model.predict_proba(X.iloc[idx_test])[:, 1]
print("Métricas en el test externo de este fold:")
print({m: round(v, 3) for m, v in classification_metrics(y[idx_test], probs_test, threshold).items()})

## 3. Validación cruzada repetida y anidada

El procedimiento de la sección 2 se repite en los 5 folds externos, y todo se repite 5 veces con particiones distintas: **25 estimaciones por comparativa**. Repetir con otras particiones muestra cuánto depende el resultado de cómo se dividieron los datos. Un solo split de 80/20 puede variar varios puntos de AUC solo por azar.

Métricas por fold:
- **AUC:** no depende del umbral. Es la métrica principal.
- **Exactitud balanceada, sensibilidad, especificidad y F1-macro:** usan el umbral elegido en la validación interna de cada fold.

Se reporta la media ± DE de los 25 folds y un IC 95 % del AUC por bootstrap. El bootstrap remuestrea sujetos sobre la probabilidad fuera de fold promediada entre repeticiones.

1Y_vs_2Y se evalúa igual que las demás comparativas. Su contraste contra el azar está en el notebook 03.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

resultados_cv = {}
for comparativa, clases in COMPARATIVAS.items():
    X, y, groups = prepare_comparison(df, demografia, clases)
    pipeline = build_pipeline(reference_class=REFERENCIA_AJUSTE[comparativa])
    resultados_cv[comparativa] = evaluate_nested(X, y, groups, pipeline, GRID_LR, N_REPEATS, N_FOLDS, SEED)
    print(f"{comparativa}: {len(y)} imágenes ({y.sum()} de clase 1), {len(resultados_cv[comparativa]['folds'])} folds evaluados")

filas = []
for comparativa, resultados in resultados_cv.items():
    resumen = summarize(resultados, comparativa)
    resumen.pop('Modelo')
    filas.append({'Comparativa': comparativa, 'Imágenes': len(resultados['y']), **resumen})
tabla_metricas = pd.DataFrame(filas).set_index('Comparativa')
tabla_metricas.to_excel('Tabla_Metricas_Modelo.xlsx')
display(tabla_metricas)

AUC de cada uno de los 25 folds externos (un punto por fold). La dispersión muestra cuánto cambia el resultado según la partición; la línea negra es la media que se reporta y la línea punteada, el azar (0.5).

In [ ]:
plot_auc_by_fold(resultados_cv)

Media de las cinco métricas en cada comparativa (la línea negra es ± 1 desviación estándar). La línea punteada marca 0.5, el valor esperado por azar. En la sensibilidad y la especificidad, la clase positiva es la cohorte conversora (en 1Y_vs_2Y, la cohorte 2Y).

In [ ]:
plot_metrics_panel(resultados_cv)

Hiperparámetros elegidos en los folds externos (frecuencia). Una elección estable indica que el grid cubre la zona adecuada.

In [ ]:
for comparativa, resultados in resultados_cv.items():
    print(f"--- {comparativa}")
    print(resultados['folds']['Hiperparametros'].astype(str).value_counts().head(4).to_string())

### 3.1 Curvas ROC y matrices de confusión

- **ROC:** curva media de los 25 folds externos; la diagonal punteada es el azar.
- **Matriz de confusión:** agrega las predicciones fuera de fold. Cada imagen se clasifica una vez por repetición, con el umbral de su fold. Cada celda muestra el conteo promedio por repetición y, entre paréntesis, el porcentaje respecto a su clase real (fila): especificidad en la fila superior y sensibilidad en la inferior.

La primera celda guarda una figura por comparativa, sin mostrarla. Las dos siguientes muestran las tres comparativas juntas.

In [ ]:
# Una figura por comparativa: se guardan sin mostrarse
rutas = []
for comparativa, resultados in resultados_cv.items():
    rutas.append(plot_mean_roc({comparativa_label(comparativa): resultados}, filename=f"ROC_{comparativa}", show=False))
    rutas.append(plot_oof_confusion(resultados, CLASES_NOMBRES[comparativa], comparativa, show=False))
print("Figuras guardadas:")
print('\n'.join(rutas))

In [ ]:
# Curva ROC media de las tres comparativas en una sola figura
plot_mean_roc({comparativa_label(c): r for c, r in resultados_cv.items()}, filename="ROC_Comparativas")

In [ ]:
# Matrices de confusión de las tres comparativas
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
for ax, (comparativa, resultados) in zip(axes, resultados_cv.items()):
    plot_oof_confusion(resultados, CLASES_NOMBRES[comparativa], comparativa, ax=ax)
save_figure("Matrices_Confusion")

## 4. Modelo final

Para 1Y_vs_Ctrl y 2Y_vs_Ctrl se entrena un modelo final con **todas** las imágenes de la comparativa:
- Los hiperparámetros se eligen con un `GridSearchCV` de 5 folds agrupado por sujeto sobre todos los datos.
- El umbral se elige por F1-macro sobre las predicciones fuera de fold de esa misma validación.

El rendimiento que se espera de este modelo es el estimado en la sección 3. El modelo final no tiene datos propios de prueba: es el que se exporta y se interpreta.

1Y_vs_2Y no tiene modelo final, porque se descarta por falta de poder predictivo (notebook 03).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

def ajustar_modelo_final(X, y, groups, pipeline, param_grid, seed):
    """Hiperparámetros por CV agrupada sobre todos los datos; umbral por F1-macro fuera de fold."""
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1).fit(X, y, groups=groups)
    model = search.best_estimator_
    probs = cross_val_predict(clone(model), X, y, groups=groups, cv=cv, method='predict_proba')[:, 1]
    return model, search.best_params_, best_threshold(y, probs)

modelos_finales, filas = {}, []
for comparativa in ['1Y_vs_Ctrl', '2Y_vs_Ctrl']:
    X, y, groups = prepare_comparison(df, demografia, COMPARATIVAS[comparativa])
    model, params, threshold = ajustar_modelo_final(X, y, groups, build_pipeline(reference_class=0), GRID_LR, SEED)
    modelos_finales[comparativa] = {'model': model, 'params': params, 'threshold': threshold, 'X': X, 'y': y, 'groups': groups}
    filas.append({'Comparativa': comparativa, 'Imágenes': len(y), 'C Lasso': params['seleccion__C'],
                  'C regresión logística': params['clasificador__C'], 'Umbral': threshold,
                  'Variables en el modelo': len(selected_features(model))})

tabla_modelos_finales = pd.DataFrame(filas).set_index('Comparativa')
tabla_modelos_finales.to_excel('Tabla_Modelos_Finales.xlsx')
display(tabla_modelos_finales)

## 5. Estabilidad de las variables (bootstrap)

Con unas 300 imágenes, Lasso puede elegir variables distintas ante pequeños cambios en los datos, sobre todo entre variables correlacionadas. Para medirlo, el pipeline del modelo final (mismos hiperparámetros) se reentrena en 500 remuestreos bootstrap de **sujetos**. Por cada variable se registra:
- **Frecuencia de selección:** fracción de remuestreos en los que Lasso la conserva.
- **Coeficiente medio** y su IC 95 % (vale 0 en los remuestreos en que no se seleccionó).

Una variable con frecuencia alta y un IC que no cruza el 0 es un hallazgo robusto. Una que aparece en el modelo final pero con frecuencia baja depende de la muestra concreta. Los coeficientes están en la escala del `RobustScaler`, así que son comparables entre variables; positivo favorece la conversión.

Las tablas y figuras nombran cada variable con su región en español y el descriptor de PyRadiomics.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

def coeficientes(model):
    classifier = model.named_steps['clasificador']
    return pd.Series(classifier.coef_[0], index=classifier.feature_names_in_)

def ajuste_bootstrap(model, X, y, groups, seed):
    """Reentrena el pipeline en un remuestreo bootstrap de sujetos y devuelve sus coeficientes."""
    rng = np.random.default_rng(seed)
    subjects = np.unique(groups)
    idx = np.concatenate([np.flatnonzero(groups == s) for s in rng.choice(subjects, len(subjects), replace=True)])
    return coeficientes(clone(model).fit(X.iloc[idx], y[idx]))

def nombrar_variables(index):
    """Región (FreeSurfer y español) y descriptor (PyRadiomics y español) de cada variable."""
    partes = [split_feature(c) for c in index]
    return pd.DataFrame({'Region_FreeSurfer': [NOMBRES_REGION.get(l, '') for l, _ in partes],
                         'Region': [NOMBRES_REGION_ES.get(l, '') for l, _ in partes],
                         'Descriptor': [d for _, d in partes],
                         'Descriptor_ES': [DESCRIPTOR_ES.get(d, d) for _, d in partes]}, index=index)

estabilidad = {}
for comparativa, final in modelos_finales.items():
    coefs = Parallel(n_jobs=-1)(delayed(ajuste_bootstrap)(final['model'], final['X'], final['y'], final['groups'], SEED + b)
                                for b in range(N_BOOT))
    matriz = pd.DataFrame(coefs).fillna(0.0)
    tabla = pd.DataFrame({'Frecuencia_Seleccion': (matriz != 0).mean(), 'Coef_Medio': matriz.mean(),
                          'Coef_IC_Inf': matriz.quantile(0.025), 'Coef_IC_Sup': matriz.quantile(0.975)})
    tabla['En_Modelo_Final'] = tabla.index.isin(coeficientes(final['model']).index)
    estabilidad[comparativa] = nombrar_variables(tabla.index).join(tabla).sort_values('Frecuencia_Seleccion', ascending=False)

    print(f"=== {comparativa}: {len(coeficientes(final['model']))} variables en el modelo final; "
          f"{(tabla['Frecuencia_Seleccion'] >= 0.5).sum()} seleccionadas en ≥ 50 % de los remuestreos, "
          f"{(tabla['Frecuencia_Seleccion'] >= 0.8).sum()} en ≥ 80 %")
    display(estabilidad[comparativa].drop(columns=['Region_FreeSurfer', 'Descriptor_ES']).head(15).round(3))

### 5.1 Coeficientes y frecuencia de selección

Las 15 variables seleccionadas con más frecuencia en el bootstrap de cada modelo, con el nombre de la región en español y el descriptor de PyRadiomics:
- **Barras de la izquierda:** coeficiente medio. Hacia la derecha (rojo), un valor alto de la variable aumenta la probabilidad de conversión; hacia la izquierda (azul), la disminuye.
- **Barras de la derecha:** frecuencia de selección en los remuestreos.

El intervalo de confianza de cada coeficiente y si la variable pertenece al modelo final están en `Tabla_Estabilidad_Bootstrap.xlsx`.

In [ ]:
plot_coefficients(estabilidad['1Y_vs_Ctrl'], NOMBRES_REGION_ES, '1Y_vs_Ctrl')

In [ ]:
plot_coefficients(estabilidad['2Y_vs_Ctrl'], NOMBRES_REGION_ES, '2Y_vs_Ctrl')

## 6. Eliminación progresiva de variables

¿Cuántas variables hacen falta realmente? Para responderlo sin optimismo, la eliminación se hace **dentro de cada fold externo** de la validación anidada:

1. Se ajusta el pipeline completo en el train (validación interna incluida). Lasso deja *n* variables.
2. Se entrena una regresión logística (con la `C` elegida) sobre esas *n* variables, se evalúa en el test y **se elimina la de menor |coeficiente|**. Se repite hasta quedar con 1 variable (eliminación recursiva, RFE).
3. Cada tamaño *k* se evalúa en el test externo del fold.

Como *n* cambia entre folds, la curva reporta el rendimiento con **hasta *k* variables**: un fold con menos de *k* variables usa todas las suyas. El AUC es la métrica principal. La exactitud balanceada usa el umbral 0.5, que es el punto natural de una regresión logística con clases balanceadas; recalcular el umbral para cada *k* multiplicaría el costo.

Para elegir el tamaño se usa la **regla de 1 error estándar**: el modelo más pequeño cuyo AUC medio no es más de un error estándar menor que el del mejor *k*.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

def eliminacion_en_fold(pipeline, param_grid, X, y, groups, idx_train, idx_test, seed):
    """Curva de métricas en el test externo al eliminar, una a una, las variables de menor |coeficiente|."""
    model, _, _ = tune_in_fold(pipeline, param_grid, X.iloc[idx_train], y[idx_train], groups[idx_train], seed)
    preprocesamiento = model[:-1]                       # ajuste, filtros y Lasso, ajustados con el train
    Z_train, Z_test = preprocesamiento.transform(X.iloc[idx_train]), preprocesamiento.transform(X.iloc[idx_test])
    C = model.named_steps['clasificador'].C
    features, filas = list(Z_train.columns), []
    while features:
        classifier = LogisticRegression(C=C, class_weight='balanced', max_iter=5000).fit(Z_train[features], y[idx_train])
        probs = classifier.predict_proba(Z_test[features])[:, 1]
        filas.append({'k': len(features), **classification_metrics(y[idx_test], probs, 0.5)})
        features.pop(int(np.argmin(np.abs(classifier.coef_[0]))))
    return pd.DataFrame(filas)

def curva_eliminacion(tablas_fold):
    """Media y error estándar por k; un fold con menos de k variables aporta su modelo completo."""
    filas = []
    for k in range(1, max(t['k'].max() for t in tablas_fold) + 1):
        valores = pd.DataFrame([t[t['k'] == min(k, t['k'].max())].iloc[0] for t in tablas_fold])
        fila = {'k': k}
        for metric in METRICS:
            fila[f'{metric}_media'] = valores[metric].mean()
            fila[f'{metric}_EE'] = valores[metric].std() / np.sqrt(len(valores))
        filas.append(fila)
    return pd.DataFrame(filas)

curvas, curvas_resumen, filas_resumen = {}, {}, []
for comparativa in ['1Y_vs_Ctrl', '2Y_vs_Ctrl']:
    X, y, groups = prepare_comparison(df, demografia, COMPARATIVAS[comparativa])
    pipeline = build_pipeline(reference_class=0)
    tablas_fold = Parallel(n_jobs=-1)(
        delayed(eliminacion_en_fold)(pipeline, GRID_LR, X, y, groups, idx_train, idx_test, SEED + 100 * r + f)
        for r, f, idx_train, idx_test in outer_splits(y, groups, N_REPEATS, N_FOLDS, SEED))
    curva = curva_eliminacion(tablas_fold)
    curvas[comparativa] = curva

    mejor = curva.loc[curva['AUC_media'].idxmax()]
    k_1se = int(curva.loc[curva['AUC_media'] >= mejor['AUC_media'] - mejor['AUC_EE'], 'k'].min())
    elegido = curva[curva['k'] == k_1se].iloc[0]
    filas_resumen.append({'Comparativa': comparativa,
                          'Variables (mediana por fold)': int(np.median([t['k'].max() for t in tablas_fold])),
                          'k con mejor AUC': int(mejor['k']), 'AUC (mejor k)': round(mejor['AUC_media'], 3),
                          'k regla 1-EE': k_1se, 'AUC (k 1-EE)': round(elegido['AUC_media'], 3),
                          'Exactitud balanceada (k 1-EE)': round(elegido['Exactitud_Balanceada_media'], 3)})
    curvas_resumen[comparativa] = (int(mejor['k']), k_1se)

tabla_eliminacion = pd.DataFrame(filas_resumen).set_index('Comparativa')
with pd.ExcelWriter('Tabla_Eliminacion_Variables.xlsx') as writer:
    tabla_eliminacion.to_excel(writer, sheet_name='Resumen')
    for comparativa, curva in curvas.items():
        curva.round(4).to_excel(writer, sheet_name=comparativa, index=False)
display(tabla_eliminacion)

Curvas de eliminación de las dos comparativas: AUC medio en los folds de prueba según el número de variables que se le permiten al modelo.
- **Punto blanco:** el número de variables con el AUC máximo.
- **Punto negro:** el modelo compacto, el más pequeño cuyo AUC queda a menos de un error estándar del máximo.

Se muestran hasta 50 variables; a partir de ahí las curvas son planas, porque casi ningún fold tiene tantas. La exactitud balanceada y el error estándar de cada punto están en `Tabla_Eliminacion_Variables.xlsx`. Se guarda además una figura por comparativa.

In [ ]:
K_MAX_FIGURA = 50
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
rutas = []
for ax, (comparativa, curva) in zip(axes, curvas.items()):
    k_mejor, k_1se = curvas_resumen[comparativa]
    plot_feature_elimination(curva, k_mejor, k_1se, comparativa, k_max_display=K_MAX_FIGURA, ax=ax)
    # La misma gráfica como figura individual: se guarda sin mostrarse
    rutas.append(plot_feature_elimination(curva, k_mejor, k_1se, comparativa, k_max_display=K_MAX_FIGURA, show=False))
plt.figure(fig.number)
save_figure("Eliminacion_Variables")
print("Figuras individuales guardadas:")
print('\n'.join(rutas))

### 6.1 Variables relevantes de los modelos finales

Se aplica la misma eliminación recursiva al **modelo final** (todos los datos) para ordenar sus variables de la más a la menos importante. Las primeras *k* (el tamaño de la regla 1-EE) forman el **modelo compacto**.

`Tabla_Variables_Relevantes.xlsx` reúne, para cada modelo final, todas sus variables con:
- el orden de importancia;
- el nombre de la región en español y en FreeSurfer;
- el descriptor de PyRadiomics y su significado;
- el coeficiente, la frecuencia de selección en el bootstrap y si pertenece al modelo compacto.

Este listado describe los modelos; su rendimiento esperado es el de la sección 3 (modelo completo) y el de la curva de la sección 6 (modelo compacto).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

def ranking_eliminacion(model, X, y):
    """Variables del modelo, de la más a la menos importante, según el orden de eliminación recursiva."""
    Z = model[:-1].transform(X)
    C = model.named_steps['clasificador'].C
    features, eliminadas = list(Z.columns), []
    while len(features) > 1:
        classifier = LogisticRegression(C=C, class_weight='balanced', max_iter=5000).fit(Z[features], y)
        eliminadas.append(features.pop(int(np.argmin(np.abs(classifier.coef_[0])))))
    return features + eliminadas[::-1]

variables_relevantes = {}
for comparativa, final in modelos_finales.items():
    ranking = ranking_eliminacion(final['model'], final['X'], final['y'])
    k = min(int(tabla_eliminacion.loc[comparativa, 'k regla 1-EE']), len(ranking))
    tabla = nombrar_variables(ranking)
    tabla.insert(0, 'Orden', range(1, len(ranking) + 1))
    tabla['Coeficiente'] = coeficientes(final['model']).reindex(ranking).to_numpy()
    tabla['Frecuencia_Seleccion'] = estabilidad[comparativa]['Frecuencia_Seleccion'].reindex(ranking).fillna(0).to_numpy()
    tabla['Modelo_Compacto'] = tabla['Orden'] <= k
    variables_relevantes[comparativa] = tabla
    print(f"=== {comparativa}: {len(ranking)} variables en el modelo final; modelo compacto de {k}")
    display(tabla.drop(columns='Region_FreeSurfer').head(max(k, 10)).round(3))

with pd.ExcelWriter('Tabla_Variables_Relevantes.xlsx') as writer:
    for comparativa, tabla in variables_relevantes.items():
        tabla.rename_axis('Variable').to_excel(writer, sheet_name=comparativa)

Los dos modelos compactos comparten varias de sus variables. La figura compara, para cada una, cuánto se aleja cada cohorte de conversión de Ctrl: la diferencia de medias dividida entre la DE de Ctrl. Valores positivos indican que la variable es mayor en la cohorte de conversión. Se usan los valores normalizados, antes del ajuste por edad y sexo.

In [ ]:
compartidas = [v for v in variables_relevantes['1Y_vs_Ctrl'].index[variables_relevantes['1Y_vs_Ctrl']['Modelo_Compacto']]
               if v in variables_relevantes['2Y_vs_Ctrl'].index[variables_relevantes['2Y_vs_Ctrl']['Modelo_Compacto']]]
ctrl = df[df['Target'] == 0]
diferencias = {nombre: {v: (df.loc[df['Target'] == t, v].mean() - ctrl[v].mean()) / ctrl[v].std() for v in compartidas}
               for t, nombre in [(1, '1Y'), (2, '2Y')]}
etiquetas = {v: f"{NOMBRES_REGION_ES[split_feature(v)[0]]}\n{split_feature(v)[1]}" for v in compartidas}
print("Variables compartidas por los dos modelos compactos:", *[e.replace(chr(10), ' · ') for e in etiquetas.values()], sep='\n')
plot_shared_variables(diferencias, etiquetas)

## 7. Exportación y verificación de los modelos

Cada modelo final se exporta como un `RadiomicsPipeline` (`ml_core/inference.py`), que reproduce todo el procesamiento a partir del CSV crudo de PyRadiomics de un paciente:
1. Exige las regiones anatómicas del entrenamiento.
2. Aplana los datos y conserva solo las características de forma.
3. Normaliza por el volumen total segmentado.
4. Recibe la edad y el sexo al escaneo, porque el modelo está ajustado por estas covariables.
5. Aplica el pipeline de sklearn y el umbral del modelo final.

La verificación procesa el CSV crudo de **todas** las imágenes de cada comparativa con el pipeline exportado y compara su probabilidad con la del modelo en memoria. Son imágenes de entrenamiento, así que esto comprueba la equivalencia numérica, no la generalización (esa la mide la sección 3).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

os.makedirs(DIR_EXPORTACION, exist_ok=True)
labels_validos = sorted({int(c.split('_')[1]) for c in df.columns if c.startswith('Label_')})

for comparativa, final in modelos_finales.items():
    ruta = os.path.join(DIR_EXPORTACION, f'pipeline_{comparativa}.joblib')
    export_pipeline(final['model'], final['threshold'], labels_validos, list(final['X'].columns), ruta, covariates=COVARIATES)

    pipeline_cargado = joblib.load(ruta)
    filas = demografia[df['Target'].isin(COMPARATIVAS[comparativa])].reset_index(drop=True)
    probs_memoria = final['model'].predict_proba(final['X'])[:, 1]
    diferencias = []
    for i, fila in filas.iterrows():
        df_crudo = pd.read_csv(os.path.join(RUTAS_COHORTES[fila['Target']], f"{fila['PatientID']}.csv"))
        salida = pipeline_cargado.predict(df_crudo, edad=fila['Edad'], sexo=fila['Sexo'])
        diferencias.append(abs(salida['Probabilidad'] - probs_memoria[i]))
    print(f"[{comparativa}] {len(diferencias)} imágenes verificadas. Diferencia máxima de probabilidad: {max(diferencias):.2e}")
    display(pipeline_cargado.get_feature_importances().head(10))